Pip installs for Oracle 

In [2]:
# pip install gymnasium[atari]
# pip install gymnasium[accept-rom-license]
# pip install stable-baselines3

In [9]:
import gymnasium as gym
import ale_py
import numpy as np
import tensorboard
from stable_baselines3 import PPO
from gymnasium.wrappers import TransformAction
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

In [4]:
# Map the RAM addresses to the physical traits of the system
class Pong6InputWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(
            low=-255, high=255, shape=(6,), dtype=np.float32
        )
        self.prev_ball_pos = None

    def observation(self, obs):
        # Cast to float32 immediately to prevent uint8 overflow during subtraction
        p1_y = float(obs[51])
        p2_y = float(obs[50])
        b_x  = float(obs[49])
        b_y  = float(obs[54])

        if self.prev_ball_pos is None:
            v_x, v_y = 0.0, 0.0
        else:
            v_x = b_x - self.prev_ball_pos[0]
            v_y = b_y - self.prev_ball_pos[1]
            
            # Logic Guard: If the ball jumps more than 10 pixels in one frame, 
            # it's a reset/teleport, not actual physics. Zero it out.
            if abs(v_x) > 10 or abs(v_y) > 10:
                v_x, v_y = 0.0, 0.0

        self.prev_ball_pos = (b_x, b_y)
        return np.array([p1_y, p2_y, b_x, b_y, v_x, v_y], dtype=np.float32)

    def reset(self, **kwargs):
        self.prev_ball_pos = None
        return super().reset(**kwargs)

In [16]:
from stable_baselines3.common.vec_env import DummyVecEnv

# 1. Standardize the action mapping
def action_mapper(a):
    return [0, 2, 3][a]

# 2. Simplified Environment Creator
def make_env():
    base_env = gym.make("ALE/Pong-v5", obs_type="ram")
    env = Pong6InputWrapper(base_env)
    env = TransformAction(env, func=action_mapper, 
                          action_space=gym.spaces.Discrete(3))
    return env

if __name__ == "__main__":
    # 3. Create 4 vectorized environments in a single process
    # DummyVecEnv still gives the Oracle 4 streams of data to learn from simultaneously,
    # which is the key to preventing that "Policy Collapse" we saw earlier.
    env = DummyVecEnv([make_env for _ in range(4)])

    # 4. The Optimized Hyper-Oracle Model
    model = PPO(
        "MlpPolicy",
        env,
        n_steps=1024,
        batch_size=256,
        learning_rate=2e-4, 
        ent_coef=0.01,
        policy_kwargs=dict(net_arch=[64, 64]),
        verbose=1,
        tensorboard_log="./ppo_pong_hyper_oracle/"
    )

    print("Training the Hyper-Oracle with 4-way vectorized data...")
    model.learn(total_timesteps=2_000_000, log_interval=25)
    model.save("pong_oracle_dnn")

Using cpu device
Training the Hyper-Oracle with 4-way vectorized data...
Logging to ./ppo_pong_hyper_oracle/PPO_2
-----------------------------------------
| time/                   |             |
|    fps                  | 898         |
|    iterations           | 25          |
|    time_elapsed         | 114         |
|    total_timesteps      | 102400      |
| train/                  |             |
|    approx_kl            | 0.005154039 |
|    clip_fraction        | 0.0207      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.976      |
|    explained_variance   | 0.46        |
|    learning_rate        | 0.0002      |
|    loss                 | 0.00684     |
|    n_updates            | 240         |
|    policy_gradient_loss | -0.00386    |
|    value_loss           | 0.0785      |
-----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 875         |
|   

In [17]:
# 1. Register and setup the environment (re-using your 6-input wrapper logic)
gym.register_envs(ale_py)

def evaluate_oracle(model_path, num_episodes=10):
    # Re-create the exact same environment stack
    base_env = gym.make("ALE/Pong-v5", obs_type="ram")
    env = Pong6InputWrapper(base_env) # Assumes the wrapper class is in scope
    env = TransformAction(env, 
                            lambda a: [0, 2, 3][a],
                            action_space=gym.spaces.Discrete(3))

    # Load the trained model
    model = PPO.load(model_path)
    
    all_episode_rewards = []
    print(f"Evaluating {model_path} over {num_episodes} episodes...")

    for episode in range(num_episodes):
        obs, info = env.reset()
        terminated = False
        truncated = False
        episode_reward = 0
        
        while not (terminated or truncated):
            # We use deterministic=True to get the 'best' performance
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
        
        all_episode_rewards.append(episode_reward)
        print(f"Episode {episode + 1}: Reward = {episode_reward}")

    mean_reward = np.mean(all_episode_rewards)
    std_reward = np.std(all_episode_rewards)
    
    print("-" * 30)
    print(f"Evaluation Results for {model_path}:")
    print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")
    print(f"Win Rate: {(mean_reward + 21) / 42 * 100:.1f}% (Approx relative to Atari AI)")
    print("-" * 30)

    env.close()
    return mean_reward

evaluate_oracle("pong_oracle_dnn", num_episodes=10)

Evaluating pong_oracle_dnn over 10 episodes...
Episode 1: Reward = 12.0
Episode 2: Reward = 8.0
Episode 3: Reward = -1.0
Episode 4: Reward = 16.0
Episode 5: Reward = 1.0
Episode 6: Reward = 4.0
Episode 7: Reward = 10.0
Episode 8: Reward = 14.0
Episode 9: Reward = 8.0
Episode 10: Reward = 6.0
------------------------------
Evaluation Results for pong_oracle_dnn:
Mean Reward: 7.80 +/- 5.19
Win Rate: 68.6% (Approx relative to Atari AI)
------------------------------


np.float64(7.8)